In [275]:
import pandas as pd
from pathlib import Path
import random 
import numpy as np
import time 
from statsmodels.stats.contingency_tables import StratifiedTable

In [218]:
pd.set_option('display.max_colwidth', None)

In [219]:
dict = pd.read_table(r"C:\Users\fidge\Documents\Project Serve\data_dictionary.txt")

dict

,## -points- columns
0,match_id - unique match identifier (see -matches- files)
1,"Pt - point number of the match, starting from 1"
2,"Set1/Set2 - number of sets won by Player1/Player2 so far in match. (Note: ""Player1"" is always the player who served first.)"
3,Gm1/Gm2 - number of games won by Player1/Player2 so far in this set.
4,Pts - point score of the current game
5,"Gm# - In the form 'X (Y)', where X = the game number of the entire match, starting from 1, and Y = the point number in this game. (For some matches that were charted earlier in the project's history, the form is simply 'X'."
6,"TbSet - '1' if the set is to be determined by a tiebreak, '0' otherwise."
7,"TB? - '1' if the point is part of a tiebreak, '0' otherwise."
8,"TBpt - point number of the tiebreak; blank if ""TB?"" == 0."
9,"Svr - '1' if Player1 is serving, '2' otherwise."


In [220]:
# Data Cleaning Process

In [221]:
pts = pd.read_csv(r"C:\Users\fidge\Documents\Project Serve\charting-m-points-2020s.csv", low_memory = False)
pts.shape

(547478, 14)

In [222]:
pts_raw = pts.copy()

In [223]:
pts_raw.shape

(547478, 14)

In [224]:
pts[pts['TbSet'].apply(type) == float]

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
531229,20200219-M-Rio_de_Janeiro-R16-Carlos_Alcaraz-Federico_Coria,157,1,1,2,2.0,15-0,25,NaN,1,0n,5f29r2f3b3b3b2f2f2f2f2f2b+;3b1v1*,NaN,1


In [225]:
pts['TbSet'] = pts['TbSet'].fillna(False).astype(bool)

In [226]:
pts['TbSet'].apply(type).value_counts()

TbSet
<class 'bool'>    547478
Name: count, dtype: int64

In [227]:
pts.dtypes

match_id        str
Pt            int64
Set1          int64
Set2          int64
Gm1           int64
Gm2         float64
Pts             str
Gm#           int64
TbSet          bool
Svr           int64
1st             str
2nd             str
Notes           str
PtWinner      int64
dtype: object

In [228]:
pd.set_option('display.max_rows', 500)

In [229]:
# pts.head(100)
game = ['20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng']
pts[pts["match_id"].isin(game)].sort_values(by = 'Pt')

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
50,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,1,0,0,0,0.0,0-0,1,True,1,6*,NaN,NaN,1
51,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,2,0,0,0,0.0,15-0,1,True,1,6f39b3f1r2f3s2f3s3f1r1f1*,NaN,NaN,2
52,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,3,0,0,0,0.0,15-15,1,True,1,4w,6b17f1n@,NaN,2
53,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,4,0,0,0,0.0,15-30,1,True,1,4n,5b28f3b3f3b3f3b3b3b3b3f3x@,NaN,1
54,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,5,0,0,0,0.0,30-30,1,True,1,6w,6b29b2b3b3b2f3b3y1*,NaN,1
55,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,6,0,0,0,0.0,40-30,1,True,1,6f29f2f1f1f1f1f1f3b3y3n@,NaN,NaN,2
56,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,7,0,0,0,0.0,40-40,1,True,1,4#,NaN,NaN,1
57,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,8,0,0,0,0.0,AD-40,1,True,1,6f18f3s1f-3*,NaN,NaN,1
58,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,9,0,0,1,0.0,0-0,2,True,2,6d,6f38b3b2f2f3b1f3b3y3b1l1o3*,NaN,2
59,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Michael_Zheng,10,0,0,1,0.0,15-0,2,True,2,6d,4b39b2f1f1d@,NaN,1


In [230]:
pts["Gm2"].isna().sum()
pts[pts["Gm2"].isna()].head(20)

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
127668,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,161,1,1,0,NaN,0-0,26,True,2,6d,4f2b3+s1v3*,NaN,2


In [231]:
game2 = ['20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer']
pts[pts["match_id"].isin(game2)].sort_values(by = 'Pt')

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
127508,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,1,0,0,0,0.0,0-0,1,True,1,6w,6w,NaN,2
127509,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,2,0,0,0,0.0,0-15,1,True,1,6d,4b3b;2f2f3b3b2n@,NaN,2
127510,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,3,0,0,0,0.0,0-30,1,True,1,4n,5b3b3b3b2n@,NaN,2
127511,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,4,0,0,0,0.0,0-40,1,True,1,5d,4b2f2d@,NaN,2
127512,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,5,0,0,0,1.0,0-0,2,True,2,6b1r3b3+m2o2f1v3m2o1f#,NaN,NaN,1
127513,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,6,0,0,0,1.0,0-15,2,True,2,c5b2b2f2+b2z1f1d@,NaN,NaN,1
127514,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,7,0,0,0,1.0,0-30,2,True,2,6n,6f2b2f1n@,NaN,2
127515,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,8,0,0,0,1.0,15-30,2,True,2,6*,NaN,NaN,2
127516,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,9,0,0,0,1.0,30-30,2,True,2,6n,6f1f3*,NaN,2
127517,20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer,10,0,0,0,1.0,40-30,2,True,2,4b2b2f1f3b2b1n@,NaN,NaN,1


In [232]:
pts = pts.sort_values(["match_id", "Pt"]).reset_index(drop=True)

pts["set_idx"] = pts["Set1"] + pts["Set2"] + 1   # sets completed = which set we're in

In [233]:
games = (pts.groupby(["match_id", "set_idx", "Gm#"], as_index=False).agg(game_winner=("PtWinner", "last"),last_pt=("Pt", "max")))

In [234]:
games = games.sort_values(["match_id", "last_pt"])

In [235]:
games["won1"] = (games["game_winner"] == 1).astype(int)
games["won2"] = (games["game_winner"] == 2).astype(int)

g = games.groupby(["match_id", "set_idx"])
games["exp_Gm1"] = g["won1"].cumsum() - games["won1"]
games["exp_Gm2"] = g["won2"].cumsum() - games["won2"]

In [236]:
chk = pts.merge(games[["match_id", "set_idx", "Gm#", "exp_Gm1", "exp_Gm2"]], on = ["match_id", "set_idx", "Gm#"], how="left")

chk["mismatch"] = (chk["Gm1"] != chk["exp_Gm1"]) | (chk["Gm2"] != chk["exp_Gm2"])

print(chk["mismatch"].sum(), "mismatched rows")
print(chk.loc[chk["mismatch"], "match_id"].nunique(), "affected matches")

chk.loc[chk["mismatch"]].groupby("match_id").size().sort_values(ascending=False).head(20)

18 mismatched rows
1 affected matches


match_id
20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer    18
dtype: int64

In [237]:
pts["PtWinner"].value_counts(dropna=False)

PtWinner
1    275853
2    271625
Name: count, dtype: int64

In [238]:
pts = pts[pts["match_id"] != "20241119-M-Montemar_CH-R32-Francesco_Passaro-Nicolai_Budkov_Kjaer"]
# Match dropped since it there was an error in the Gm2 column which reset the Gm2 counter casuing a downstream error for the rest of the match.

In [239]:
pts['Gm2'] = pts['Gm2'].astype("int64")


In [240]:
pts = pts.rename(columns = {'Gm#' : 'Gm_num'})

In [241]:
dup = pts.duplicated(["match_id", "Pt"], keep=False)
print(dup.sum(), "duplicated rows")
print(pts.loc[dup, "match_id"].nunique(), "affected matches")

pts.groupby("match_id")["Pt"].agg(["size", "nunique"]).query("size != nunique")

560 duplicated rows
2 affected matches


,size,nunique
match_id,,
20220812-M-Canada_Masters-QF-Hubert_Hurkacz-Nick_Kyrgios,408,204
20230227-M-Dubai-R32-Alejandro_Davidovich_Fokina-Malek_Jaziri,152,76


In [242]:
d = pts[dup].sort_values(["match_id", "Pt"])
d.duplicated(keep=False).sum()   # exact full-row duplicates

np.int64(166)

In [243]:
pts = pts.drop_duplicates(subset=["match_id", "Pt"], keep="first")

In [244]:
svr_per_game = (pts.groupby(["match_id", "set_idx", "Gm_num"])["Svr"].transform("nunique"))
pts["is_tiebreak"] = svr_per_game > 1

In [245]:
print(pts["is_tiebreak"].sum())

20614


In [246]:
pts[pts["is_tiebreak"]].groupby(["match_id","set_idx","Gm_num"]).ngroups

1691

In [247]:
first = pts[pts["is_tiebreak"]].iloc[0]
pts[(pts["match_id"] == first["match_id"]) &
   (pts["Gm_num"] == first["Gm_num"])][["match_id","Pt","Svr", "Gm1", "Gm2"]]

,match_id,Pt,Svr,Gm1,Gm2
143,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,144,1,6,6
144,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,145,2,6,6
145,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,146,2,6,6
146,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,147,1,6,6
147,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,148,1,6,6
148,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,149,2,6,6
149,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,150,2,6,6
150,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,151,1,6,6
151,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,152,1,6,6
152,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,153,2,6,6


In [248]:
tb_games = (pts[pts["is_tiebreak"]]
            .groupby(["match_id","set_idx","Gm_num"])
            .agg(n_pts=("Pt","size"), gm1=("Gm1","first"), gm2=("Gm2","first")))

tb_games[["gm1","gm2"]].value_counts().head(10) 
#flagging games which are tie breaks 

gm1  gm2
6    6      1671
3    3        19
0    0         1
Name: count, dtype: int64

In [249]:
tb_games["n_pts"].min() #check for real tie breaks 

np.int64(7)

In [250]:
# Data analysis

In [251]:
import sqlite3

In [252]:
con = sqlite3.connect("tennis.db")

pts.to_sql("points", con, if_exists = "replace", index = False)

con.commit()
con.close()

In [253]:
con = sqlite3.connect("tennis.db")
df = pd.read_sql("SELECT * FROM points WHERE Svr = 1 LIMIT 10", con)
cur = con.cursor()
cur.execute("CREATE INDEX IF NOT EXISTS idx_match_pt ON points(match_id, Pt)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_match_gm ON points(match_id, Gm_num)")
con.commit()
con.close()

In [254]:
df.columns

Index(['match_id', 'Pt', 'Set1', 'Set2', 'Gm1', 'Gm2', 'Pts', 'Gm_num',
       'TbSet', 'Svr', '1st', '2nd', 'Notes', 'PtWinner', 'set_idx',
       'is_tiebreak'],
      dtype='str')

In [255]:
%%writefile sql/analysis_table.sql
WITH flagged AS (
    SELECT
        match_id,
        set_idx,
        Gm_num,
        Pt,
        Pts,
        Svr,
        CASE WHEN PtWinner = Svr THEN 1 ELSE 0 END AS server_won
    FROM points
    WHERE is_tiebreak = 0
)
SELECT
    match_id,
    set_idx,
    Gm_num,
    Pt,
    Pts,
    Svr,
    server_won,
    LAG(server_won) OVER (
        PARTITION BY match_id, set_idx, Gm_num
        ORDER BY Pt
    ) AS prev_server_won
FROM flagged;

Overwriting sql/analysis_table.sql


In [256]:
query = Path("sql/analysis_table.sql").read_text()

with sqlite3.connect("tennis.db") as con:
    analysis = pd.read_sql(query, con)

print(analysis.shape)

analysis.head(10)

(526406, 8)


,match_id,set_idx,Gm_num,Pt,Pts,Svr,server_won,prev_server_won
0,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,1,0-0,1,0,NaN
1,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,2,0-15,1,0,0.0
2,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,3,0-30,1,1,0.0
3,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,4,15-30,1,0,1.0
4,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,5,15-40,1,1,0.0
5,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,1,6,30-40,1,0,1.0
6,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,2,7,0-0,2,1,NaN
7,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,2,8,15-0,2,1,1.0
8,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,2,9,30-0,2,1,1.0
9,20200103-M-ATP_Cup-RR-Alex_De_Minaur-Alexander_Zverev,1,2,10,40-0,2,1,1.0


In [257]:
print(analysis["prev_server_won"].isna().sum())           # should equal the number of games

86011


In [258]:
analysis.groupby(["match_id","set_idx","Gm_num"]).ngroups
print(analysis["server_won"].mean())  # should be roughly 0.63-0.66 for men

0.6426541490788479


In [259]:
seq = analysis.dropna(subset = ["prev_server_won"]) #Dropping the first point of every game

In [260]:
len(analysis) - len(seq) #confirming that it equals the game count.

86011

In [261]:
seq["prev_server_won"] = seq["prev_server_won"].astype(int) 
#Cleaning up the data by changing column from float to int after NaN has been dropped

In [262]:
seq.dtypes

match_id             str
set_idx            int64
Gm_num             int64
Pt                 int64
Pts                  str
Svr                int64
server_won         int64
prev_server_won    int64
dtype: object

In [263]:
naive = seq.groupby("prev_server_won")["server_won"].agg(["mean", "count", "sum"])
naive

,mean,count,sum
prev_server_won,,,
0,0.631503,171076,108035
1,0.649178,269319,174836


In [264]:
diff = naive.loc[1, "mean"] - naive.loc[0, "mean"]
print(diff)

0.017675154451743613


In [265]:
naive["se"] = np.sqrt(naive["mean"] * (1 - naive["mean"]) / naive["count"])

In [266]:
naive

,mean,count,sum,se
prev_server_won,,,,
0,0.631503,171076,108035,0.001166
1,0.649178,269319,174836,0.000920


In [267]:
#Permutation test (starting form the full analysis table)
full = analysis.sort_values(["match_id","set_idx","Gm_num","Pt"]).reset_index(drop = True)
# Creating a unique integerID for each distinct combination of game, set and match 
g, _ = pd.factorize(
    full["match_id"] + "|" + full["set_idx"].astype(str) + "|" + full["Gm_num"].astype(str)
)
y = full["server_won"].to_numpy(np.int64)
n = len(y)

#pair (i-1,i) is usable only if both points are in the same game.
same = g[1:] == g[:-1]

In [268]:
# The test statistic P(server wins| server wins previous point) - P(sever wins| server lost previous point)
def stat(y_arr):
    cur = y_arr[1:][same]
    prev = y_arr[:-1][same]
    n1 = prev.sum()
    n0 = prev.size - n1
    s1 = (cur * prev).sum()
    s0 = cur.sum() - s1
    return s1 / n1 - s0 / n0
obs = stat(y)
print(obs)

0.017675154451743613


In [269]:
rng = np.random.default_rng(42)

def shuffled_stat():
    order = np.argsort(g + rng.random(n)) # sorts by group, random within group 
    return stat(y[order])
    

In [270]:
#testing how long the output will take before commiting.
t = time.time()
for i in range(100):
    shuffled_stat()
print((time.time() - t) / 100 * 10000 / 60, "minutes projected")

10.162907044092814 minutes projected


In [271]:
B = 10000
null = np.empty(B)
for i in range(B):
    null[i] = shuffled_stat()

centre = null.mean()
p = (1 + np.sum(np.abs(null - centre) >= abs(obs - centre))) / (1 + B)

print("observed: ", obs ,"\nnull mean: ", null.mean(), "\nnull sd: ", null.std(), "\np-val: ", p)

observed:  0.017675154451743613 
null mean:  0.05664999333327706 
null sd:  0.0012132666687200116 
p-val:  9.999000099990002e-05


In [272]:
seq.groupby(["Pts", "prev_server_won"])["server_won"].agg(["mean", "count"])


mean  count
Pts   prev_server_won                 
0-15  0                0.626394  30583
0-30  0                0.616280  11425
0-40  0                0.596395   4383
15-0  1                0.644889  55425
15-15 0                0.649477  19682
      1                0.651877  19157
15-30 0                0.622070  13566
      1                0.614827   7041
15-40 0                0.619546   7838
      1                0.612089   2614
30-0  1                0.672682  35742
30-15 0                0.645153  11698
      1                0.648372  25271
30-30 0                0.649536  13037
      1                0.636357  12768
30-40 0                0.624729   9212
      1                0.609975   6456
40-0  1                0.677414  24037
40-15 0                0.664001   7753
      1                0.673548  23930
40-30 0                0.643371  10417
      1                0.634545  16593
40-40 0                0.630921  18156
      1                0.625493  17762
40-AD 0                0.605583  13326
AD-40 1                0.628025  22523

In [291]:
# Creating a table of wins/losses per score state, split by previous point outcome
tab = seq.groupby(["Pts","prev_server_won"])["server_won"].agg(["sum","count"])
tab["fail"] = tab["count"] - tab["sum"]

#Loops through each score state, building a 2x2 table
strata, per = [], []
for state, sub in tab.groupby(level=0):
    if state in ("40-40", "AD-40","40-AD"):
        continue
    if sub.index.get_level_values(1).nunique() < 2:
        continue                      # skips data with only one of the two previous point outcomes
    a, b = sub.loc[(state, 1), ["sum", "fail"]]
    c, d = sub.loc[(state, 0), ["sum", "fail"]]
    strata.append(np.array([[a, b], [c, d]], dtype = float))
    p1, p0 = a/ (a + b) , c / (c + d) # win rate difference and its uncertainty per score state.
    se = np.sqrt(p1 * (1 - p1) / (a + b) + p0 * (1 - p0) / (c + d))
    per.append({"Pts": state, "p_prev_win": p1, "p_prev_loss": p0, "diff": p1 - p0,
               "se": se, "n": int(a + b + c + d)})
per = pd.DataFrame(per)

#Overall estimate from pooling the per-score states
w = 1 / per["se"] ** 2
pooled = (per["diff"] * w).sum() / w.sum()
se_pooled = np.sqrt(1 / w.sum())

In [292]:
print(per.round(4))

     Pts  p_prev_win  p_prev_loss    diff      se      n
0  15-15      0.6519       0.6495  0.0024  0.0048  38839
1  15-30      0.6148       0.6221 -0.0072  0.0071  20607
2  15-40      0.6121       0.6195 -0.0075  0.0110  10452
3  30-15      0.6484       0.6452  0.0032  0.0053  36969
4  30-30      0.6364       0.6495 -0.0132  0.0060  25805
5  30-40      0.6100       0.6247 -0.0148  0.0079  15668
6  40-15      0.6735       0.6640  0.0095  0.0062  31683
7  40-30      0.6345       0.6434 -0.0088  0.0060  27010


In [293]:
print("pooled diff:", round(pooled, 5), "se:", round(se_pooled, 5),
      "z:", round(pooled / se_pooled, 2))

pooled diff: -0.00289 se: 0.00221 z: -1.31


In [294]:
print("95% CI:", (round(pooled - 1.96*se_pooled, 5), round(pooled + 1.96*se_pooled, 5)))

95% CI: (np.float64(-0.00723), np.float64(0.00145))


In [295]:
# Cochran-Mantel-Haenszel test
st = StratifiedTable(np.dstack(strata))
print(st.summary())

                   Estimate   LCB    UCB 
-----------------------------------------
Pooled odds           0.987   0.968 1.006
Pooled log odds      -0.013  -0.032 0.006
Pooled risk ratio     0.995              
                                         
                 Statistic P-value 
-----------------------------------
Test of OR=1         1.845   0.174 
Test constant OR    13.224   0.067 
                         
-------------------------
Number of tables      8  
Min n             10452  
Max n             38839  
Avg n             25879  
Total n          207033  
-------------------------


In [296]:
print(st.test_null_odds())

pvalue      0.17430946329760633
statistic   1.8454847148467541
